# Table of Contents
* [Section 1: Imports & Configuration](#imports)
* [Section 2: Extracting Words With Their Timecodes](#extract-words)
* [Section 3: Mapping Words To Phonemes](#map-words)
* [Section 4: Rhyme Unit Extraction](#extract-rhyme-units)
* [Section 5: Pairwise Rhyme Similarity](#pairwise-similarity)
* [Section 6: Words Clustering](#clustering)
* [Section 7: HTML Generation](#html-generation)

---
## Section 1: Imports & Configuration <a id="imports"></a>
---

In [1]:
import sys
sys.path.append("..")
print(sys.executable)

from python.rhyme_extraction import extract_rhyme_unit, extract_end_words
from python.similarity_engine import (
    rhyme_similarity,
    compute_similarity_pairs,
    build_similarity_matrix
)
from python.clustering import (
    SIMILARITY_THRESHOLD, 
    cluster_rhymes, 
    cluster_exact_rhymes
)
from python.debugging import debug_rhyme_analysis
from python.evaluation import evaluate_rhyme_detection
from python.html_generation import generate_rhyme_html

from praatio import textgrid

RHYME_MODE = "stressed_plus"
# Options for RHYME_MODE:
# "stressed"
# "stressed_plus"
# "entire_word"

INPUT_TXT_PATH = "input/current_input/input.txt"
OUTPUT_TEXTGRID_PATH = "output/current_output/output.TextGrid"

C:\Users\georg\anaconda3\envs\mfa_env\python.exe


---
## Section 2: Extracting Words With Their Timecodes <a id="extract-words"></a>
---

In [2]:
tg = textgrid.openTextgrid(OUTPUT_TEXTGRID_PATH, includeEmptyIntervals=False)

word_tier = tg.getTier("words")
phone_tier = tg.getTier("phones")

for word in word_tier.entries:
    print(word)

Interval(start=1.68, end=1.82, label='i')
Interval(start=1.82, end=2.24, label='found')
Interval(start=2.24, end=2.3, label='the')
Interval(start=2.3, end=2.81, label='sound')
Interval(start=2.93, end=3.1, label='of')
Interval(start=3.1, end=3.21, label='the')
Interval(start=3.21, end=4.03, label='underground')
Interval(start=4.34, end=4.43, label='the')
Interval(start=4.43, end=4.89, label='crowd')
Interval(start=4.89, end=5.14, label='was')
Interval(start=5.14, end=5.57, label='loud')
Interval(start=5.62, end=5.84, label='and')
Interval(start=5.84, end=6.33, label='proud')
Interval(start=6.33, end=6.85, label='around')
Interval(start=6.9, end=7.0, label='the')
Interval(start=7.0, end=7.55, label='mound')
Interval(start=8.08, end=8.17, label='a')
Interval(start=8.17, end=8.58, label='light')
Interval(start=8.66, end=8.86, label='at')
Interval(start=8.86, end=9.24, label='night')
Interval(start=9.24, end=10.04, label='ignites')
Interval(start=10.15, end=10.31, label='the')
Interval(sta

---
## Section 3: Mapping Words To Phonemes <a id="map-words"></a>
---

In [3]:
for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    phonemes = []
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    print(word_label + ":", phonemes)

# Populating the word_to_phonemes dictionary
word_to_phonemes = {}

for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    
    phonemes = []
    
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    
    word_label = word_label.lower()
    
    if word_label not in word_to_phonemes:
        word_to_phonemes[word_label] = phonemes

print(word_to_phonemes)

i: ['AY1']
found: ['F', 'AW1', 'N', 'D']
the: ['DH', 'AH0']
sound: ['S', 'AW1', 'N', 'D']
of: ['AH1', 'V']
the: ['DH', 'IY0']
underground: ['AH1', 'N', 'D', 'ER0', 'G', 'R', 'AW2', 'N', 'D']
the: ['DH', 'AH0']
crowd: ['K', 'R', 'AW1', 'D']
was: ['W', 'AH0', 'Z']
loud: ['L', 'AW1', 'D']
and: ['AE1', 'N', 'D']
proud: ['P', 'R', 'AW1', 'D']
around: ['ER0', 'AW1', 'N', 'D']
the: ['DH', 'AH0']
mound: ['M', 'AW1', 'N', 'D']
a: ['AH0']
light: ['L', 'AY1', 'T']
at: ['AE1', 'T']
night: ['N', 'AY1', 'T']
ignites: ['IH0', 'G', 'N', 'AY1', 'T', 'S']
the: ['DH', 'AH0']
kite: ['K', 'AY1', 'T']
in: ['IH0', 'N']
flight: ['F', 'L', 'AY1', 'T']
the: ['DH', 'AH0']
sight: ['S', 'AY1', 'T']
of: ['AH0', 'V']
white: ['W', 'AY1', 'T']
delight: ['D', 'IH0', 'L', 'AY1', 'T']
shines: ['SH', 'AY1', 'N', 'Z']
bright: ['B', 'R', 'AY1', 'T']
tonight: ['T', 'AH0', 'N', 'AY1', 'T']
{'i': ['AY1'], 'found': ['F', 'AW1', 'N', 'D'], 'the': ['DH', 'AH0'], 'sound': ['S', 'AW1', 'N', 'D'], 'of': ['AH1', 'V'], 'underground': 

---
## Section 4: Rhyme Unit Extraction <a id="extract-rhyme-units"></a>
---

In [4]:
rhyme_candidates = extract_end_words(INPUT_TXT_PATH, word_to_phonemes, rhyme_mode=RHYME_MODE)

for r in rhyme_candidates:
    print(r)

{'line_index': 0, 'line_text': 'I found the sound of the underground', 'end_word': 'underground', 'phonemes': ['AH1', 'N', 'D', 'ER0', 'G', 'R', 'AW2', 'N', 'D'], 'rhyme_unit': ['AH1', 'N', 'D', 'ER0', 'G', 'R', 'AW2', 'N', 'D']}
{'line_index': 1, 'line_text': 'The crowd was loud and proud around the mound', 'end_word': 'mound', 'phonemes': ['M', 'AW1', 'N', 'D'], 'rhyme_unit': ['AW1', 'N', 'D']}
{'line_index': 2, 'line_text': 'A light at night ignites the kite in flight', 'end_word': 'flight', 'phonemes': ['F', 'L', 'AY1', 'T'], 'rhyme_unit': ['AY1', 'T']}
{'line_index': 3, 'line_text': 'The sight of white delight shines bright tonight', 'end_word': 'tonight', 'phonemes': ['T', 'AH0', 'N', 'AY1', 'T'], 'rhyme_unit': ['AY1', 'T']}


---
## Section 5: Pairwise Rhyme Similarity <a id="pairwise-similarity"></a>
---

In [5]:
similarity_pairs = compute_similarity_pairs(rhyme_candidates, rhyme_mode=RHYME_MODE)

for pair in similarity_pairs:
    print(pair)

# Building a similarity matrix
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)

for w1 in similarity_matrix:
    print(w1, similarity_matrix[w1])

{'wordA': 'underground', 'wordB': 'mound', 'similarity': 1.0}
{'wordA': 'underground', 'wordB': 'flight', 'similarity': 0.0}
{'wordA': 'underground', 'wordB': 'tonight', 'similarity': 0.0}
{'wordA': 'mound', 'wordB': 'flight', 'similarity': 0.0}
{'wordA': 'mound', 'wordB': 'tonight', 'similarity': 0.0}
{'wordA': 'flight', 'wordB': 'tonight', 'similarity': 1.0}
underground {'underground': 1.0, 'mound': 1.0, 'flight': 0.0, 'tonight': 0.0}
mound {'underground': 1.0, 'mound': 1.0, 'flight': 0.0, 'tonight': 0.0}
flight {'underground': 0.0, 'mound': 0.0, 'flight': 1.0, 'tonight': 1.0}
tonight {'underground': 0.0, 'mound': 0.0, 'flight': 1.0, 'tonight': 1.0}


---
## Section 6: Words Clustering <a id="clustering"></a>
---

In [6]:
# Clustering by thresholds from the similarity matrix
cluster_labels, clusters = cluster_rhymes(similarity_matrix, threshold=SIMILARITY_THRESHOLD) # SIMILARITY_THRESHOLD defined in 'clustering.py'

print("Clusters:")
print(clusters)

print("\nWord → Cluster Mapping:")
print(cluster_labels)

# Defining expected_clusters and comparing with the currect clustering
expected_clusters = {
    "underground": "A",
    "mound": "A",
    "flight": "B",
    "tonight": "B"
}

for r in rhyme_candidates:
    word = r["end_word"]
    r["expected_cluster"] = expected_clusters.get(word, None)

for r in rhyme_candidates:
    print(r)

# Exact rhymes clusters printout
clusters = cluster_labels

print(clusters)

# Debugging the clustering
debug_rhyme_analysis(rhyme_candidates, clusters)

# Accuracy evaluation
accuracy, mismatches = evaluate_rhyme_detection(rhyme_candidates, clusters)

Clusters:
{'A': ['underground', 'mound'], 'B': ['flight', 'tonight']}

Word → Cluster Mapping:
{'underground': 'A', 'mound': 'A', 'flight': 'B', 'tonight': 'B'}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'end_word': 'underground', 'phonemes': ['AH1', 'N', 'D', 'ER0', 'G', 'R', 'AW2', 'N', 'D'], 'rhyme_unit': ['AH1', 'N', 'D', 'ER0', 'G', 'R', 'AW2', 'N', 'D'], 'expected_cluster': 'A'}
{'line_index': 1, 'line_text': 'The crowd was loud and proud around the mound', 'end_word': 'mound', 'phonemes': ['M', 'AW1', 'N', 'D'], 'rhyme_unit': ['AW1', 'N', 'D'], 'expected_cluster': 'A'}
{'line_index': 2, 'line_text': 'A light at night ignites the kite in flight', 'end_word': 'flight', 'phonemes': ['F', 'L', 'AY1', 'T'], 'rhyme_unit': ['AY1', 'T'], 'expected_cluster': 'B'}
{'line_index': 3, 'line_text': 'The sight of white delight shines bright tonight', 'end_word': 'tonight', 'phonemes': ['T', 'AH0', 'N', 'AY1', 'T'], 'rhyme_unit': ['AY1', 'T'], 'expected_cluster': 'B'

In [7]:
print(clusters)

{'underground': 'A', 'mound': 'B', 'flight': 'C', 'tonight': 'C'}


---
## Section 7: HTML Generation <a id="html-generation"></a>
---

In [8]:
clusters = clusters = cluster_labels

generate_rhyme_html(rhyme_candidates, clusters)

HTML file generated: generated_html/rhyme_visualization.html
